In [ ]:
!pip install ultralytics

In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET
from tqdm import tqdm
import glob
import random
import yaml
from ultralytics import YOLO

print("Đã cài đặt và import thành công!")

In [ ]:

INPUT_DIR = '/kaggle/input/datasets/andrewmvd/face-mask-detection'
WORKING_DIR = '/kaggle/working'

# Tạo cấu trúc folder YOLO trong /kaggle/working
for split in ['train', 'val']:
    for sub in ['images', 'labels']:
        os.makedirs(f'{WORKING_DIR}/{split}/{sub}', exist_ok=True)

classes = ["with_mask", "without_mask", "mask_weared_incorrect"]
print("Đã tạo xong cấu trúc thư mục rỗng.")

In [ ]:
def process_xml_to_yolo(xml_path):
    """Chuyển đổi XML sang YOLO format và kiểm tra class thiểu số"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find('size')
    w, h = int(size.find('width').text), int(size.find('height').text)
    
    yolo_lines = []
    is_minority = False 
    empha = False
    
    for obj in root.findall('object'):
        label = obj.find('name').text
        if label not in classes: continue
        cls_id = classes.index(label)
        
        if cls_id in [1, 2]: is_minority = True 
        if cls_id in [2] : empha = True
            
        xmlbox = obj.find('bndbox')
        xn, xx = float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text)
        yn, yx = float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text)
        
        # Tọa độ chuẩn hóa YOLO
        x_center = (xn + xx) / (2.0 * w)
        y_center = (yn + yx) / (2.0 * h)
        width = (xx - xn) / w
        height = (yx - yn) / h
        yolo_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
        
    return yolo_lines, is_minority, empha

print("Đã định nghĩa hàm xử lý xong.")

In [ ]:
# --- CELL THIẾU: THỰC THI ĐỔ DỮ LIỆU ---
all_xmls = glob.glob(f'{INPUT_DIR}/annotations/*.xml')

if len(all_xmls) == 0:
    print("Lỗi: Không tìm thấy file XML. Hãy kiểm tra lại path INPUT_DIR!")
else:
    random.seed(42)
    random.shuffle(all_xmls)
    # Chia 80/20
    split_idx = int(len(all_xmls) * 0.8)
    
    dataset_splits = {
        'train': all_xmls[:split_idx],
        'val': all_xmls[split_idx:]
    }

    for split_name, xml_list in dataset_splits.items():
        print(f"--- Đang đổ dữ liệu vào tập {split_name} ---")
        for xml_path in tqdm(xml_list):
            file_id = os.path.basename(xml_path).replace('.xml', '')
            
            # Kiểm tra ảnh (Kaggle dataset này thường là .png)
            img_src = os.path.join(INPUT_DIR, 'images', file_id + '.png')
            if not os.path.exists(img_src):
                img_src = img_src.replace('.png', '.jpg')
            
            if not os.path.exists(img_src): continue
            
            # Gọi hàm xử lý tọa độ và kiểm tra Upsampling
            yolo_lines, is_minority, empha = process_xml_to_yolo(xml_path)
            
            # UPSAMPLING: Gấp 3 lần class thiểu số ở tập TRAIN
            if (is_minority and split_name == 'train' and not empha):
                repeat = 3
            elif (is_minority and split_name == 'train' and empha):
                repeat = 5
            else:
                repeat = 1

            
            for i in range(repeat):
                new_id = f"{file_id}_up{i}" if i > 0 else file_id
                # 1. Copy ảnh vào folder images tương ứng
                shutil.copy(img_src, f'{WORKING_DIR}/{split_name}/images/{new_id}.png')
                # 2. Ghi nội dung nhãn vào folder labels tương ứng
                with open(f'{WORKING_DIR}/{split_name}/labels/{new_id}.txt', 'w') as f:
                    f.write('\n'.join(yolo_lines))

    print(f"Đã đổ dữ liệu thành công! Bạn có thể chuyển sang bước Train.")

In [ ]:
data_yaml = {
    'path': '/kaggle/working',
    'train': 'train/images',
    'val': 'val/images',
    'names': {i: name for i, name in enumerate(classes)}
}

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print(" Đã tạo file data.yaml")

In [ ]:

model = YOLO('/kaggle/input/models/tngngtun/bestmodel/pytorch/default/1/yolov8n.pt')


model.train(
    data='/kaggle/working/data.yaml',
    epochs=100,
    imgsz=640,
    device=0, # Ép sử dụng GPU
    project='/kaggle/working/FaceMaskRuns',
    name='v8_mask_upsampled'
)

In [ ]:
# 1. Tải mô hình tốt nhất vừa train xong
# Đường dẫn này phụ thuộc vào tham số 'project' và 'name' bạn đặt lúc train
best_model_path = '/kaggle/working/FaceMaskRuns/v8_mask_upsampled/weights/best.pt'
model = YOLO(best_model_path)

# 2. Chạy đánh giá (Validation)
# YOLO sẽ tự động sử dụng tập 'val' đã khai báo trong data.yaml
metrics = model.val()

# 3. Hiển thị các chỉ số tổng quát
print(f"--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---")
print(f"mAP@50: {metrics.box.map50:.4f}")       # Độ chính xác trung bình tại IoU 0.5
print(f"mAP@50-95: {metrics.box.map:.4f}")     # mAP trung bình trên nhiều ngưỡng IoU
print(f"Precision (Trung bình): {metrics.box.mp:.4f}")
print(f"Recall (Trung bình): {metrics.box.mr:.4f}")